# Step 1, scored — temporal input and the normalisation fix, separated

`claude_temporal_train` produced checkpoints at `temporal_radius` 0 and 1, selected on
**paired recall**. This scores them on the real metric, against the champion.

## Four arms, and why not three

| arm | radius | `prob_input_norm` | what it isolates |
|---|---|---|---|
| `champion` | — | — | drift control, reproduces CV 0.7070 in-run |
| `r0_movie` | 0 | `movie` | the notes/21–22 serving path, reproduced |
| `r0_perframe` | 0 | `per_frame` | **minus `r0_movie` = the normalisation skew** |
| `r1_perframe` | 1 | `per_frame` | **minus `r0_perframe` = temporal input** |

Two changes landed between `notes/22` and here: the temporal input, and a train/serve
normalisation fix. Scoring `r1_perframe` against `r0_movie` alone would bundle them and
credit both to the temporal input. That is the `notes/18` §1 failure — and it is *harder*
to catch here, because the bundled number would look better than either change deserves.

So `r0_perframe` exists purely to split them. It costs one arm.

## What the normalisation bug was

Every training notebook stored `dog_response(load_frame(...))[0]` as its input tensor — a
**per-frame** percentile rescale. `predict_dataset` handed `prob_fn` the raw `load_frame`
output, normalised by **whole-movie** quantiles from the zarr attrs. Different
distributions: on the test volumes the two differ by 5.7× in range.

So every learned arm ever scored through this pipeline — `notes/21`, `notes/22`, and the
submission now sitting on the leaderboard — was served an input distribution it had never
trained on. Those runs are internally consistent with each other, because they all had it.
`r0_movie` reproduces that path so the comparison is exact rather than remembered.

## The gate

`adaptive_predicted`, CV **0.7070**, the configuration behind the 0.752 leaderboard score.
It is reproduced in-run so drift is charged to the champion and not to the new arm.

## Pre-registered

1. **Fixing the normalisation skew is worth something.** Until now every learned arm trained on a per-frame percentile rescale and was SERVED the whole-movie one -- 5.7x off in range on the test volumes. Measured as: r0_perframe scores above r0_movie. Falsified if it does not, which would mean the network is insensitive to input scale and the skew was never costing anything.
2. **Temporal input beats single-frame at matched normalisation, through coherence.** notes/22 sized the remaining coherence deficit at 88% of 66.5 points. Measured as: r1_perframe beats r0_perframe on SCORE **and** on temporal position. Falsified if the score moves without the position moving -- that would mean something other than coherence produced it and the mechanism is still not understood.
3. **It is still not enough to pass the champion gate.** notes/22 projected 0.7006 at full DoG-parity coherence from a 0.6556 baseline -- short of the champion's 0.7128 edge Jaccard unless edge PRECISION moves too. Recorded so that a pass draws scrutiny rather than celebration. Falsified if any learned arm scores above 0.7070.

In [ ]:
import subprocess, sys, time

def sh(*args, **kw):
    try:
        return subprocess.run(args, capture_output=True, text=True, **kw)
    except (FileNotFoundError, OSError) as e:
        return subprocess.CompletedProcess(args, 127, "", str(e))

def pip_install(pkgs, extra=()):
    r = sh(sys.executable, "-m", "pip", "install", "-q", *extra, *pkgs)
    if r.returncode != 0:
        print(r.stdout[-2000:]); print(r.stderr[-2000:])
    return r.returncode == 0

gpu = sh("nvidia-smi", "--query-gpu=name", "--format=csv,noheader").stdout.strip()
print(f"accelerator: {gpu or 'NONE'}")
if "P100" in gpu:
    print("P100 -> installing torch with sm_60 kernels ...")
    t0 = time.time()
    ok = pip_install(["torch==2.5.1"],
                     extra=("--index-url", "https://download.pytorch.org/whl/cu121"))
    print(f"  torch replacement {'ok' if ok else 'FAILED'} ({time.time()-t0:.0f}s)")
print("installing geff + zarr ...")
pip_install(["geff", "zarr"])

In [ ]:
import sys, os, gc, time, json, hashlib
from pathlib import Path
import numpy as np
import torch

WORK = Path("/kaggle/working"); WORK.mkdir(parents=True, exist_ok=True)
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__}  device {DEV}")
if DEV.type == "cuda":
    try:
        _w = torch.nn.Conv3d(1, 4, 3, padding=1).to(DEV)
        _ = _w(torch.randn(2, 1, 8, 8, 8, device=DEV)).sum().item()
        torch.cuda.synchronize(); print("  GPU smoke test passed")
    except Exception as e:
        raise SystemExit(f"GPU present but unusable: {type(e).__name__}: {str(e)[:200]}")

def find_dir(is_match, roots, max_depth=5):
    for root in roots:
        root = Path(root)
        if not root.is_dir():
            continue
        stack = [(root, 0)]
        while stack:
            d, depth = stack.pop(0)
            try:
                if is_match(d):
                    return d
                if depth >= max_depth:
                    continue
                kids = [e for e in d.iterdir()
                        if e.is_dir() and e.suffix not in (".zarr", ".geff")]
            except (PermissionError, OSError):
                continue
            stack += [(k, depth + 1) for k in kids]
    return None

REPO = find_dir(lambda p: (p / "harness").is_dir() and (p / "pipeline").is_dir(),
                [WORK, "/kaggle/input"])
if REPO is None:
    raise SystemExit("Could not find harness/ and pipeline/.")
sys.path.insert(0, str(REPO))

from harness import Harness, gate
import pipeline.classical as pc
for field in ("refine", "temporal_radius", "prob_input_norm"):
    if field not in pc.Config.__dataclass_fields__:
        raise SystemExit(f"The uploaded snapshot's Config has no `{field}` — re-upload. "
                         "Without it the arms below are not what they claim to be.")
if "prob_fn" not in pc.predict_dataset.__code__.co_varnames:
    raise SystemExit("The uploaded snapshot's predict_dataset has no prob_fn — re-upload.")
from pipeline.classical import (Config, budget_features, estimated_total_nodes,
                                make_predictor, predict_dataset)
from pipeline.detector import paired_recall
from pipeline.unet import UNet3D, predict_volume
from harness.purescore import match_nodes
from harness.tracks import read_geff

# Weights arrive as a KERNEL data source. Bounded scan -- a recursive glob over
# /kaggle/input walks every .zarr chunk and costs minutes (notes/17 §4).
def find_weights(root="/kaggle/input", max_depth=5):
    found, stack = set(), [(Path(root), 0)]
    while stack:
        d, depth = stack.pop()
        try:
            kids = list(os.scandir(d))
        except (PermissionError, OSError, FileNotFoundError):
            continue
        for e in kids:
            if e.is_file() and e.name.startswith("claude_temporal_r") and e.name.endswith(".pt"):
                # resolve(): a symlinked mount reaches the same file by two paths, and an
                # unresolved set would load it twice and report phantom duplicates.
                found.add(Path(e.path).resolve())
        if depth < max_depth:
            stack += [(Path(e.path), depth + 1) for e in kids
                      if e.is_dir() and not e.name.endswith((".zarr", ".geff"))]
    return sorted(found)

wpaths = find_weights()
print(f"weight files found: {[p.name for p in wpaths] or 'NONE'}")
if not wpaths:
    raise SystemExit("No claude_temporal_r*.pt found. Add the claude_temporal_train kernel "
                     "as a data source (Add Input -> Notebook Output).")

ckpts = {}              # (radius, train_emb) -> (path, checkpoint)
for wp in wpaths:
    ck = torch.load(wp, map_location="cpu")
    key = (int(ck["temporal_radius"]), ck["train_emb"])
    if key in ckpts:
        raise SystemExit(f"Two checkpoints for {key}: {ckpts[key][0].name} and {wp.name}. "
                         "Attach exactly one training kernel version.")
    ckpts[key] = (wp, ck)
    print(f"  {wp.name}: r={key[0]}, trained on {key[1]}, "
          f"paired={ck.get('best_paired'):.4f} node={ck.get('best_recall'):.4f} "
          f"@epoch {ck.get('best_epoch')} (DoG paired {ck.get('dog_paired'):.4f})")

by_radius = {}
for (rad, emb), v in ckpts.items():
    by_radius.setdefault(rad, {})[emb] = v
embryos_needed = {e for _, e in ckpts}
# Both radii must cover both folds or the comparison is a chimera: an r=1 number built
# from one fold against an r=0 number built from two is not a difference in radius.
missing = {rad: sorted(embryos_needed - set(v)) for rad, v in by_radius.items()}
missing = {k: v for k, v in missing.items() if v}
if missing or set(by_radius) != {0, 1}:
    raise SystemExit(
        f"Need both radii on both folds. Have "
        f"{ {r: sorted(v) for r, v in by_radius.items()} }"
        + (f"; missing {missing}" if missing else "")
        + ". The training run's sub-DoG guard refuses to save a checkpoint worse than "
          "DoG, so a gap here means that arm genuinely failed and there is nothing to "
          "score -- read the training log rather than re-running this.")

MODELS = {}             # radius -> {embryo -> model}
for rad, per_emb in sorted(by_radius.items()):
    MODELS[rad] = {}
    for emb, (wp, ck) in sorted(per_emb.items()):
        m = UNet3D(base=ck.get("base", 16), depth=ck.get("depth", 3),
                   in_ch=ck.get("in_ch", 1 if rad == 0 else 3))
        m.load_state_dict(ck["state_dict"]); m.eval().to(DEV)
        MODELS[rad][emb] = m
        print(f"  r={rad}: {wp.name} scores datasets NOT from {emb}")

COMP = find_dir(lambda p: (p / "train").is_dir() and (p / "test").is_dir()
                and any((p / "train").glob("*.zarr")), ["/kaggle/input"])
TRAIN = COMP / "train"
CACHE = WORK / "cache"; CACHE.mkdir(exist_ok=True, parents=True)
train_names = sorted({p.stem for p in TRAIN.glob("*.zarr")} & {p.stem for p in TRAIN.glob("*.geff")})

SUBSET_SIZE = 60
def stable_key(n): return int(hashlib.sha1(n.encode()).hexdigest(), 16)
by_prefix = {}
for n in train_names:
    by_prefix.setdefault(n.split("_")[0], []).append(n)
SUBSET = []
for pfx, ns in sorted(by_prefix.items()):
    SUBSET += sorted(ns, key=stable_key)[:round(SUBSET_SIZE * len(ns) / len(train_names))]
SUBSET = sorted(SUBSET)
assert len(SUBSET) == 60, f"subset drifted ({len(SUBSET)})"

h = Harness(data_dir=TRAIN, cache_dir=CACHE)
folds = {}
for n in SUBSET:
    folds.setdefault(h.fold_of(n), []).append(n)
prefixes = {f: {n.split("_")[0] for n in v} for f, v in folds.items()}
assert all(len(p) == 1 for p in prefixes.values()), "folds are NOT leave-one-embryo-out"
print("folds:", {f: len(v) for f, v in sorted(folds.items())}, prefixes)

SCALES2 = [(1.5, 4.0), (2.5, 6.0)]
BASE = dict(detector="dog", dog_rel_threshold=0.005, dog_scales=SCALES2, footprint="ball")
CHAMPION_CV = 0.7070          # notes/14: adaptive_predicted, and the 0.752 LB submission
BASELINE_CV = 0.6490          # notes/22: unet_cap1.2_norefine, the arm to beat
results = {}

## 1. The budget regression, refit here

`notes/14` §2: a constant budget scores **0.0882 below doing nothing**, so this is refit at
runtime rather than carried as coefficients. Same features and same leave-one-embryo-out
fit that measured 10.7 % median error in `08`, and reproduced exactly in `notes/21` and
`notes/22`.

In [ ]:
CFG_FEAT = Config(min_separation_um=6.0, **BASE)
FRAC_FRAMES, REF_SEPS = (0.25, 0.5, 0.75), (4.0, 8.0, 16.0)
FEAT_NAMES = ["n_sep4", "n_sep8", "n_sep16", "nstrong_sep4", "nstrong_sep8",
              "nstrong_sep16", "mean_int", "frac_fg"]

t0 = time.time()
FEATS, BUDGETS = {}, {}
for i, n in enumerate(SUBSET):
    FEATS[n] = budget_features(TRAIN / f"{n}.zarr", CFG_FEAT,
                               frac_frames=FRAC_FRAMES, ref_seps=REF_SEPS)
    BUDGETS[n] = estimated_total_nodes(TRAIN / f"{n}.zarr")
    if (i + 1) % 20 == 0:
        print(f"  features {i+1}/{len(SUBSET)}  ({time.time()-t0:.0f}s)", flush=True)

def design(rows):
    return np.array([[1.0] + [np.log1p(f[k]) if k.startswith("n") else
                              np.log(max(f[k], 1e-6)) for k in FEAT_NAMES]
                     for f in rows], float)

usable = [n for n in SUBSET if BUDGETS.get(n)]
X = design([FEATS[n] for n in usable])
y = np.array([np.log(BUDGETS[n] / max(1.0, FEATS[n]["T"])) for n in usable])
pref = np.array([n.split("_")[0] for n in usable])

PRED_BUDGETS, errs = {}, []
for g in sorted(set(pref)):
    m = pref == g
    b_g = np.linalg.lstsq(X[~m], y[~m], rcond=None)[0]
    e = np.exp(X[m] @ b_g) * np.array([FEATS[n]["T"] for n, k in zip(usable, m) if k])
    for n, v in zip([n for n, k in zip(usable, m) if k], e):
        PRED_BUDGETS[n] = float(v)
    errs.append(np.abs(e / np.array([BUDGETS[n] for n, k in zip(usable, m) if k]) - 1))
pooled = np.concatenate(errs)
print(f"\nleave-one-embryo-out budget error: median {np.median(pooled):.1%} "
      f"(08 measured 10.7%, notes/21-22 reproduced it exactly)")

## 2. Score the arms

`diagnose` collects the near/far miss split and the paired-recall accounting per dataset,
so a result arrives with a mechanism rather than as a bare number. That is what turned
`notes/21`'s gate failure into the most informative run in the project.

In [ ]:
SCALE_UM = (1.625, 0.40625, 0.40625)     # GT lives in FULL-resolution voxels
NEAR_UM = 14.0                           # 2x the 7um match radius
DIAG, CURRENT_ARM = {}, ""

def diagnose(name, data_dir, graph, s):
    g = read_geff(Path(data_dir) / f"{name}.geff")
    if not len(g.t) or not len(graph.t):
        return
    matched = match_nodes(graph.t, graph.zyx, g.t, g.zyx, scale=s, max_distance=7.0)
    hit = set(matched[matched >= 0].tolist())
    near = far = 0
    for t in np.unique(g.t):
        gi = np.flatnonzero(g.t == t)
        miss = [i for i in gi if i not in hit]
        if not miss:
            continue
        pj = np.flatnonzero(graph.t == t)
        if not len(pj):
            far += len(miss); continue
        d = np.linalg.norm((g.zyx[miss][:, None] - graph.zyx[pj][None]) * s, axis=2).min(1)
        near += int((d <= NEAR_UM).sum()); far += int((d > NEAR_UM).sum())
    # The same coherence measure the training run selected on, now on FULL movies rather
    # than short runs -- so the number that chose the checkpoint and the number that
    # explains the score are the same quantity.
    pr = paired_recall(graph.t, graph.zyx, g.t, g.zyx, g.edges, s)
    DIAG.setdefault(CURRENT_ARM, []).append(
        {"name": name, "n_gt": int(len(g.t)), "matched": len(hit),
         "near_miss": near, "far_miss": far, "n_pred": int(len(graph.t)),
         "paired": pr["paired"], "position": pr["position"],
         "pair_edges": pr["n_edges"]})

def make_unet_predictor(cfg, budgets, radius):
    def _fn(name, data_dir):
        emb = name.split("_")[0]
        other = [e for e in MODELS[radius] if e != emb]
        model = MODELS[radius][other[0] if other else emb]
        def prob_fn(vol):
            return predict_volume(model, vol, DEV)
        graph = predict_dataset(Path(data_dir) / name, cfg, verbose=False,
                                est_total_nodes=budgets.get(name), prob_fn=prob_fn)
        diagnose(name, data_dir, graph, SCALE_UM)
        return graph
    return _fn

def make_dog_predictor(cfg, budgets):
    base = make_predictor(cfg, budgets=budgets)
    def _fn(name, data_dir):
        graph = base(name, data_dir)
        diagnose(name, data_dir, graph, SCALE_UM)
        return graph
    return _fn

def run(name, cfg, predictor=None):
    global CURRENT_ARM
    CURRENT_ARM = name
    t0 = time.time()
    fn = predictor if predictor is not None else make_dog_predictor(cfg, PRED_BUDGETS)
    res = h.evaluate(fn, arm=name, names=SUBSET, verbose=False)
    s = res.summary
    n = sum(r["num_pred_nodes"] for r in res.rows.values())
    mult = s["adj_edge_jaccard"] / s["edge_jaccard"] if s["edge_jaccard"] else float("nan")
    rows = DIAG.get(name, [])
    tot_e = sum(r["pair_edges"] for r in rows)
    pos = (sum(r["position"] * r["pair_edges"] for r in rows
               if r["position"] == r["position"]) / tot_e) if tot_e else float("nan")
    print(f"{name:<16} SCORE={s['score']:.4f}  edge_J={s['edge_jaccard']:.4f}  "
          f"mult={mult:.4f}  recall={s['node_recall']:.3f}  nodes={n:>9,}  "
          f"position={pos:+.1%}  ({time.time()-t0:.0f}s)", flush=True)
    results[name] = res
    results[name].position = pos
    return res

# The champion, reproduced here so drift cannot be charged to the new arms.
run("champion", Config(min_separation_um=6.0, adaptive_separation=True,
                       adaptive_target=1.2, prune_isolated_nodes=True, **BASE))
drift = results["champion"].score - CHAMPION_CV
print(f"\nreproduction: {results['champion'].score:.4f} vs {CHAMPION_CV:.4f} "
      f"(drift {drift:+.4f})")
if abs(drift) > 0.005:
    print("!! the champion moved — read everything below against THIS number")

In [ ]:
# budget_fill=1.2 and refine=False are notes/22's best learned configuration, held fixed
# across all three learned arms so the only things that move are the two under test.
LEARNED = dict(min_separation_um=6.0, budget_fill=1.2, refine=False,
               prune_isolated_nodes=True, **BASE)

for tag, radius, norm in (("r0_movie", 0, "movie"),
                          ("r0_perframe", 0, "per_frame"),
                          ("r1_perframe", 1, "per_frame")):
    cfg = Config(temporal_radius=radius, prob_input_norm=norm, **LEARNED)
    run(tag, cfg, predictor=make_unet_predictor(cfg, PRED_BUDGETS, radius))

## 3. Grade the pre-registered predictions

In [ ]:
PREDICTIONS = [["Fixing the normalisation skew is worth something.", "Until now every learned arm trained on a per-frame percentile rescale and was SERVED the whole-movie one -- 5.7x off in range on the test volumes. Measured as: r0_perframe scores above r0_movie. Falsified if it does not, which would mean the network is insensitive to input scale and the skew was never costing anything.", "norm_fix_helps"], ["Temporal input beats single-frame at matched normalisation, through coherence.", "notes/22 sized the remaining coherence deficit at 88% of 66.5 points. Measured as: r1_perframe beats r0_perframe on SCORE **and** on temporal position. Falsified if the score moves without the position moving -- that would mean something other than coherence produced it and the mechanism is still not understood.", "temporal_beats_single"], ["It is still not enough to pass the champion gate.", "notes/22 projected 0.7006 at full DoG-parity coherence from a 0.6556 baseline -- short of the champion's 0.7128 edge Jaccard unless edge PRECISION moves too. Recorded so that a pass draws scrutiny rather than celebration. Falsified if any learned arm scores above 0.7070.", "still_short"]]
ref = results["champion"]

def S(tag): return results[tag].score if tag in results else float("nan")
def P(tag): return getattr(results[tag], "position", float("nan")) if tag in results else float("nan")

print(f"{'arm':<16} {'SCORE':>8} {'edge_J':>8} {'recall':>7} {'position':>10} {'vs champ':>9}")
print("-" * 62)
for tag in ("champion", "r0_movie", "r0_perframe", "r1_perframe"):
    if tag not in results:
        continue
    s = results[tag].summary
    print(f"{tag:<16} {s['score']:>8.4f} {s['edge_jaccard']:>8.4f} "
          f"{s['node_recall']:>7.3f} {P(tag):>9.1%} "
          f"{s['score'] - ref.score:>+9.4f}")

d_norm = S("r0_perframe") - S("r0_movie")
d_temp = S("r1_perframe") - S("r0_perframe")
d_pos = P("r1_perframe") - P("r0_perframe")
print(f"\nDECOMPOSED, and this is the point of the fourth arm:")
print(f"  normalisation fix   r0_movie -> r0_perframe   {d_norm:+.4f}")
print(f"  temporal input      r0_perframe -> r1_perframe {d_temp:+.4f}  "
      f"(position {d_pos:+.1%})")
print(f"  together                                       "
      f"{S('r1_perframe') - S('r0_movie'):+.4f}")
print(f"\n  notes/22 baseline (unet_cap1.2_norefine): {BASELINE_CV:.4f}")
print(f"  best learned arm here:                    "
      f"{max(S('r0_movie'), S('r0_perframe'), S('r1_perframe')):.4f}")

best_learned = max(S("r0_movie"), S("r0_perframe"), S("r1_perframe"))
VERDICTS = {
    "norm_fix_helps": bool(d_norm > 0),
    # BOTH clauses. A score gain without a position gain means something other than
    # coherence produced it, and notes/21's mechanism would still be unconfirmed -- so
    # this must not read as a confirmation.
    "temporal_beats_single": bool(d_temp > 0 and d_pos > 0),
    "still_short": bool(best_learned <= ref.score),
}

print()
for i, (claim, why, key) in enumerate(PREDICTIONS, 1):
    print(f"{i}. {'CONFIRMED' if VERDICTS[key] else 'FALSIFIED':<10} {claim}")

if not VERDICTS["still_short"]:
    print(f"\n*** A LEARNED ARM PASSED THE GATE: {best_learned:.4f} > {ref.score:.4f}. "
          "Prediction 3 was written to make this draw scrutiny rather than celebration — "
          "check the champion's drift above before believing it.")
if VERDICTS["temporal_beats_single"] is False and d_temp > 0:
    print(f"\n!! The score moved {d_temp:+.4f} but temporal position did NOT "
          f"({d_pos:+.1%}). Whatever produced the gain, it was not the mechanism "
          "notes/21 identified — do not report it as confirmation of that mechanism.")

blob = json.dumps({
    "scores": {t: results[t].score for t in results},
    "summaries": {t: dict(results[t].summary) for t in results},
    "position": {t: P(t) for t in results},
    "deltas": {"normalisation": d_norm, "temporal": d_temp, "position": d_pos},
    "champion_cv": CHAMPION_CV, "champion_drift": drift,
    "baseline_cv": BASELINE_CV, "verdicts": VERDICTS,
    "diag": {k: v for k, v in DIAG.items()},
}, indent=2, default=float)
(WORK / "claude_temporal_score_results.json").write_text(blob)
print("\nwrote claude_temporal_score_results.json")